# Laboratorio 08 — Dejar la libreta ordenada

**Duración: 60 minutos.**

Último laboratorio del curso, y el más **operativo** de todos: lo que vas a hacer hoy
es la rutina que el equipo de la DGT va a ejecutar todas las semanas, para siempre,
una vez que sus tablas estén en Iceberg.

El laboratorio pasado cerró con esto:

> Tu tabla ya está migrada y desde mañana recibe documentos todos los días.
>
> **¿Cómo se mantiene una tabla que trabaja todos los días?**

---

### En el almacén

Al almacenero le llegan pedidos a toda hora, y cada uno lo anota en un papelito
suelto. Al final del mes la información está completa —no se perdió nada— pero para
encontrar el pedido de un cliente hay que **revisar quinientos papelitos**.

Lo que hace entonces es lo obvio: **los pasa en limpio a una hoja ordenada**. La misma
información, un solo papel. Eso es **compactar**.

Y hay una segunda cosa. La libreta lleva años de páginas viejas y el cajón donde se
guardan no es infinito. En algún momento hay que **botar los borradores que ya nadie
va a pedir**. Eso es **expirar la historia**, y viene con una advertencia que hoy
vamos a decir varias veces:

> **Lo que se bota, no vuelve.** Es la única operación de todo el curso que no se
> puede deshacer.

---

### Cómo se usa este notebook

Como siempre: `%%sql` en la primera línea, tú escribes debajo, una sentencia por
celda.

Tres cosas de hoy:

- **El paso 0 es largo a propósito.** Son doce cargas, una por mes. El problema de
  hoy no te lo entregamos hecho: **lo fabricas tú**, con cargas que entiendes, que es
  exactamente como se fabrica solo en una tabla que recibe documentos todos los días.
- Vas a mirar mucho las **vistas del sistema**, así que acuérdate del **nombre a tres
  partes**: `mi_espacio.documentos_recibidos.files`, aunque ya hayas hecho `USE`.
- Hay **dos celdas que fallan a propósito**, y las dos van anunciadas.


## Paso 0 — Un año de ingesta, mes a mes

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero.

In [ ]:
%%sql
-- Celda 0.2



Ahora la primera carga del año: **enero**. La tabla se crea con ella, como en el
laboratorio 03.

Fíjate bien en la forma de la sentencia, porque las once que siguen son la misma
cambiando un número.

In [ ]:
%%sql
-- Celda 0.3



Febrero. De aquí en adelante ya no es `CREATE`: la tabla existe y le agregamos el
lote del mes con `INSERT INTO`.

In [ ]:
%%sql
-- Celda 0.4



Marzo.

In [ ]:
%%sql
-- Celda 0.5



Abril.

In [ ]:
%%sql
-- Celda 0.6



Mayo.

In [ ]:
%%sql
-- Celda 0.7



Junio.

In [ ]:
%%sql
-- Celda 0.8



Julio.

In [ ]:
%%sql
-- Celda 0.9



Agosto.

In [ ]:
%%sql
-- Celda 0.10



Septiembre.

In [ ]:
%%sql
-- Celda 0.11



Octubre.

In [ ]:
%%sql
-- Celda 0.12



Noviembre.

In [ ]:
%%sql
-- Celda 0.13



Y diciembre, la última.

In [ ]:
%%sql
-- Celda 0.14



Doce cargas, el año completo. Levantemos el **inventario**: es la medición que
vamos a repetir dos veces más, para comprobar que la mantención **no toca los
datos**.

In [ ]:
%%sql
-- Celda 0.15



## Paso 1 — Medir el costo

Treinta mil documentos. **Anota ese total.**

Ahora miremos cómo quedó guardada la tabla por dentro, con la consulta de archivos del
laboratorio 02.

In [ ]:
%%sql
-- Celda 1.1



Ahí está el problema del laboratorio, en una fila:

- **Doce archivos** para treinta mil documentos: uno por cada carga.
- Un promedio de **unos 74 mil bytes por archivo**, o sea **72 KB**.

Setenta y dos kilobytes es *nada*. Iceberg, cuando escribe, apunta a archivos de
**512 MB** —es su tamaño objetivo de fábrica—, y estos son siete mil veces más chicos.

Y no es que doce archivos sean muchos: es que **son doce después de doce cargas**. La
DGT no carga doce veces al año, carga todos los días. Al año son 365 archivos por esta
misma vía; en tres años, más de mil, y cada uno con las migajas de un día.

Veámoslos uno por uno.

In [ ]:
%%sql
-- Celda 1.2



Doce archivos, un mes cada uno, 2.500 documentos cada uno.

**Y aquí conviene comparar con el laboratorio 04, porque el problema es otro.** Allá
se trataba de *descartar*: pocos archivos leídos era bueno porque el motor podía saltarse
los demás. Eso sigue funcionando —fíjate en las columnas `desde` y `hasta`: cada archivo
cubre un mes, así que para preguntar por junio se puede descartar once—.

El problema de hoy es distinto: **cada archivo que sí hay que abrir se paga**. Abrirlo,
leer su pie de página, pedirle sus columnas. Con doce archivos de 72 KB no se nota. Con
diez mil archivos de 72 KB, el motor pasa más tiempo abriendo archivos que leyendo
documentos, y la consulta que ayer salía en un minuto hoy no sale.

Esta es la consulta con la que Valentina cierra el año, y tiene que abrirlos todos.

In [ ]:
%%sql
-- Celda 1.3



## Paso 2 — Pasar en limpio

Doce períodos, 2.500 documentos cada uno. Correcto, y para responderlo hubo que
abrir los doce archivos.

---

Pasemos los papelitos en limpio. El procedimiento se llama **`rewrite_data_files`** y
hace exactamente eso: lee los archivos chicos y **escribe archivos grandes con el mismo
contenido**.

Tres cosas antes de ejecutarlo:

- **No cambia ni un dato.** Reorganiza dónde están guardados, nada más. Lo vamos a
  comprobar con números en dos celdas.
- **No borra nada.** Los archivos viejos se quedan en el almacenamiento, y en el paso 3
  vas a ver por qué.
- Se le pasa **el nombre completo de la tabla, con la database**, entre comillas, igual
  que a `rollback_to_snapshot` en el laboratorio 01.

In [ ]:
%%sql
-- Celda 2.1



La salida dice lo que hizo:

- `rewritten_data_files_count`: **12** archivos leídos.
- `added_data_files_count`: **1** archivo escrito.
- `rewritten_bytes_count`: cuántos bytes movió.

Ahora **la misma consulta de archivos del paso 1**, letra por letra igual.

In [ ]:
%%sql
-- Celda 2.2



Compara las dos filas, que es a lo que vinimos:

| | archivos | bytes por archivo | bytes totales |
|---|---|---|---|
| **Antes** | 12 | ~74.000 | ~888.000 |
| **Después** | **1** | **~761.000** | **~761.000** |

Doce archivos convertidos en uno, y el promedio multiplicado por diez.

Y fíjate en la última columna, que sorprende a todo el mundo: **el total ocupa menos que
antes**. No se perdió nada; es que treinta mil documentos juntos en un archivo se
comprimen mejor que repartidos en doce, porque el formato Parquet comprime por bloques y
cada archivito arrastraba su propia cabecera.

La comprobación que importa: **el mismo inventario del paso 0**, sin cambiar una letra.

In [ ]:
%%sql
-- Celda 2.3



## Paso 3 — Lo que no se liberó

**Los mismos 30.000 documentos y exactamente el mismo total**, hasta el centavo.

Compactar **reorganiza, no altera**. Si alguno de esos dos números hubiera cambiado, no
sería una compactación: sería una pérdida de datos, y lo sabrías ahora.

---

Miremos las páginas de la libreta, como en el laboratorio 01.

In [ ]:
%%sql
-- Celda 3.1



**Trece páginas.** Doce `append` —tus doce cargas— y una `replace`, que es la
compactación.

Ahí hay algo que conviene notar: compactar **agregó una página**. La operación que vino
a ordenar la tabla también dejó su huella en la historia. Eso es lo correcto: en Iceberg
todo lo que toca la tabla queda anotado, incluida la mantención.

**Copia el `snapshot_id` de la PRIMERA fila** —la de más arriba, la carga de enero— y
tenlo a mano: lo vas a usar **dos veces**, y la segunda es la importante.

Donde dice `<TU_SNAPSHOT_ID>`, escribe tu número, sin comillas.

In [ ]:
%%sql
-- Celda 3.2



**2.500 documentos**: la tabla como estaba después de cargar enero, antes de los
otros once meses y antes de compactar.

O sea que **el viaje en el tiempo sigue funcionando después de compactar**. Y ahí está la
respuesta a por qué compactar no liberó espacio: para poder mostrarte esa página, la
tabla **tiene que conservar los archivos viejos**. Los doce siguen ahí.

Hay una vista del sistema que los cuenta todos: los de la página vigente y los de todas
las anteriores.

In [ ]:
%%sql
-- Celda 3.3



## Paso 4 — Botar los borradores

**Trece archivos guardados**: el archivo compactado más los doce viejos, que ya no
usa ninguna consulta de hoy pero que sostienen la historia.

Esa es la cuenta que hay que entender para operar Iceberg: **el espacio no se libera
solo**. Compactar ordena, pero deja atrás; hasta que alguien decida botar la historia,
la tabla ocupa lo nuevo *más* lo viejo.

---

> ## Lee esta celda entera antes de ejecutarla
>
> `expire_snapshots` **borra páginas de la libreta, y con ellas los archivos que solo
> esas páginas usaban**. Después de ejecutarlo:
>
> - Los estados anteriores a la fecha de corte **dejan de ser consultables**.
> - `VERSION AS OF` y `TIMESTAMP AS OF` sobre esas páginas **fallan**.
> - **No hay vuelta atrás.** No es como el `rollback` del laboratorio 01, que solo
>   cambiaba cuál página manda y dejaba todo lo demás en su lugar: esto **borra**.
>
> Es la única operación de todo el curso que destruye información.

Le vamos a pasar tres cosas:

| Argumento | Qué significa |
|---|---|
| `table` | La tabla, con su database, entre comillas |
| `older_than` | La **fecha de corte**: se bota todo lo anterior a ese instante |
| `retain_last` | Cuántas páginas conservar siempre, pase lo que pase |

La fecha de corte de hoy es **la de la compactación**: botamos la historia anterior a
haber pasado en limpio, y conservamos de ahí en adelante. Vuelve a la lista de páginas de
hace tres celdas, **copia el `committed_at` de la ÚLTIMA fila** —la `replace`— y
escríbelo entre comillas donde dice `<TU_ULTIMO_COMMITTED_AT>`, **completo, con los
milisegundos**.

In [ ]:
%%sql
-- Celda 4.1



La salida es el recibo de lo que se botó: cuántos archivos de datos, cuántos
manifiestos y cuántas listas de manifiestos —los otros dos niveles de la libreta, los del
laboratorio 02—.

Ahora la comprobación. **La misma celda de viaje en el tiempo del paso 3**, con el mismo
número que ya copiaste.

> **Esta celda va a fallar**, y es el punto del paso. Mira el mensaje: no dice *"no
> encontré datos"*, dice que **esa página no existe**.

In [ ]:
%%sql
-- Celda 4.2



`Cannot find snapshot with ID`. Esa página de la libreta **ya no está**, y no hay
sentencia que la traiga de vuelta. Hace tres minutos respondía 2.500; ahora no existe.

Así se ve la irreversibilidad, y por eso la advertencia iba **antes** y no después.

Veamos qué quedó de la historia.

In [ ]:
%%sql
-- Celda 4.3



**Una sola página**, la de la compactación, que es la que manda. Las doce anteriores
se fueron.

Y el almacenamiento, con la misma consulta del paso 3.

In [ ]:
%%sql
-- Celda 4.4



**Un archivo.** De trece a uno: los doce viejos, que nadie podía consultar ya, se
borraron de verdad. Ese es el espacio que se recupera, y es la razón por la que esta
rutina existe.

Última comprobación, la que deja tranquilo a todo el mundo: **el inventario**, por
tercera vez.

In [ ]:
%%sql
-- Celda 4.5



## Paso 5 — Los archivos que nadie reclama

**30.000 documentos y el mismo total del paso 0.** Se botó la historia; **los datos
vigentes no se tocaron**. Esa distinción es todo lo que hay que entender de esta
operación.

---

Queda una tercera rutina, y es la más delicada de las tres.

Cuando una escritura se interrumpe —el proceso se cae, alguien mata el trabajo a mitad—
pueden quedar archivos escritos en el almacenamiento que **ninguna página de la libreta
menciona**. Son los **archivos huérfanos**: ocupan espacio y no los usa nadie, y como no
están en la libreta, ni compactar ni expirar los ven.

`remove_orphan_files` los busca y los borra. La advertencia es distinta de la del paso 4:

> El procedimiento borra **todo archivo del directorio de la tabla que la tabla no
> reclame**. Si hay un proceso escribiendo en ese momento, sus archivos todavía no están
> declarados en ninguna página: para este procedimiento son huérfanos, **y se los
> lleva**.

Por eso se ejecuta con `dry_run => true`, que **no borra nada** y solo lista lo que
borraría. Es la forma en que se corre la primera vez, siempre.

In [ ]:
%%sql
-- Celda 5.1



**Ninguna fila: no hay huérfanos**, y así debe ser. Ninguna de tus escrituras se
interrumpió, así que todo archivo del directorio está declarado en la libreta.

Iceberg además trae una defensa de fábrica, y vale la pena verla funcionar. Intentemos
mirar los archivos **de la última hora**, poniendo como corte la misma fecha de la
compactación que usaste en el paso 4.

> **Esta celda también falla**, y es la última. Lee el mensaje: es el motivo de la
> defensa.

In [ ]:
%%sql
-- Celda 5.2



---

## Paso 6 — La rutina

`Cannot remove orphan files with an interval less than 24 hours`. Iceberg **se niega**
a mirar archivos de menos de 24 horas, y explica por qué: un archivo recién escrito
puede ser de un trabajo que todavía no confirma. Borrarlo corrompería esa carga.

Esa negativa resume el criterio de las tres rutinas. Así se ordenan en la operación:

| Rutina | Qué hace | Cada cuánto | Cuidado |
|---|---|---|---|
| **`rewrite_data_files`** | Junta los archivos chicos | Según el ritmo de ingesta: si se carga a diario, semanal | Ninguno: no cambia datos y se puede repetir |
| **`expire_snapshots`** | Bota la historia anterior al corte | Según lo que exija la auditoría | **Destructiva.** La fecha de corte es una decisión, no un parámetro técnico |
| **`remove_orphan_files`** | Borra lo que la tabla no reclama | Rara vez, y con `dry_run` primero | La más peligrosa. Nunca con trabajos escribiendo |

Y el orden, cuando se corren juntas: **compactar, después expirar, y los huérfanos
aparte**. Compactar primero porque es lo que deja historia vieja atrás; expirar después
porque es lo que la libera.

Tres criterios para llevarse:

- **Cuándo compactar** lo decide el ritmo de ingesta. Una tabla que recibe un lote al día
  se compacta una vez por semana; una que recibe cada hora, todos los días. La señal es
  la que mediste hoy: cantidad de archivos contra tamaño promedio.
- **Cuánta historia conservar** no lo decide el área de datos: lo decide **la exigencia de
  auditoría de la institución**. Si Contraloría puede preguntar por el estado de una tabla
  a seis meses, la historia se conserva seis meses, ocupe lo que ocupe. Esa conversación se
  tiene **antes** de programar la rutina, no el día que alguien pregunta por una página que
  ya se botó.
- **La ventana** es la de menor actividad, y nunca con cargas corriendo.

Estas tres rutinas **no se ejecutan a mano todas las semanas**: se programan con las
herramientas de orquestación de la plataforma —en la CDP de la DGT, con el planificador de
trabajos del clúster—, igual que cualquier otro proceso batch de la institución. Lo que
acabas de aprender es qué hace cada una y qué se pierde si te equivocas en la fecha de
corte.

### Una advertencia para el día que compactes una tabla particionada

La tabla de hoy no está particionada, y por eso el año entero terminó en un solo archivo.
En una tabla particionada —como las del laboratorio 04— **la compactación trabaja dentro
de cada partición**: junta los pedacitos de junio entre ellos y deja junio en uno o dos
archivos, sin mezclarlo con julio.

Es exactamente lo que uno quiere: se ganan archivos grandes **sin perder el descarte por
partición**. Pero significa que compactar una tabla de tres años de historia diaria es un
trabajo grande, y por eso se acota con filtros a las particiones recientes, que son las
que reciben las cargas chicas.

---

## Paso 7 — Cerrar el curso

Nueve laboratorios. Este cierra las preguntas que los anteriores dejaron abiertas a
propósito:

| Quedó abierto en | La pregunta | La respuesta de hoy |
|---|---|---|
| **Laboratorio 01** | Cada carga y cada corrección agregan una página, y las páginas viejas quedan. ¿Hasta cuándo? | Hasta que alguien las expire. La historia no se recorta sola, y conservarla es una decisión de auditoría |
| **Laboratorio 03** | Fusionar, corregir y borrar también dejan páginas | Las mismas rutinas las ordenan: son la misma tabla y la misma libreta |
| **Laboratorio 04** | «Particionar crea más archivos, y eso tiene un costo propio» | Ese costo es el de hoy: archivos diminutos que hay que abrir. Se paga compactando, dentro de cada partición |
| **Laboratorio 05** | Se eliminó una columna y el espacio no se liberó | Porque eliminar una columna no reescribe archivos. El espacio se recupera cuando se compacta y se expira, no al hacer el `ALTER` |
| **Laboratorio 06** | Las ramas de auditoría también acumulan páginas | Una rama viva protege sus páginas de la expiración. Por eso las ramas de trabajo se botan cuando ya cumplieron |
| **Laboratorio 07** | La migración dejó un respaldo que comparte archivos con la tabla migrada | Y por eso **`remove_orphan_files` no es un botón que se aprieta**: sobre una tabla migrada, lo que parece huérfano puede ser el dato de otra |

### Lo que te llevas del curso

| Lo que la DGT no podía hacer | Con qué se hace ahora |
|---|---|
| Saber qué decía un dato el mes pasado | `VERSION AS OF`, `TIMESTAMP AS OF` |
| Deshacer una carga mala | `CALL … rollback_to_snapshot` |
| Cargar sin duplicar | `MERGE INTO` |
| Corregir y borrar filas | `UPDATE`, `DELETE FROM` |
| Consultar un mes sin leer el año | `PARTITIONED BY (months(…))` y el particionamiento oculto |
| Cambiar el formulario sin reprocesar la historia | `ALTER TABLE … ADD COLUMNS` |
| Revisar un lote antes de publicarlo | Ramas, `REPLACE BRANCH`, etiquetas |
| Pasar las tablas Hive a Iceberg sin cerrar | `CALL … migrate`, con inventario antes y después |
| **Mantener la tabla en el tiempo** | **`rewrite_data_files`, `expire_snapshots`, `remove_orphan_files`** |

Todo eso es SQL, es el mismo que corre en la plataforma de la DGT, y lo escribiste tú.

### Lo que queda por delante

Lo del laboratorio 07 sigue siendo lo primero: **migrar una tabla de verdad**, con el
inventario levantado antes y verificado después. Y desde el día siguiente, esta rutina.

La libreta del almacén no se ordena sola. Pero ahora sabes cómo ordenarla, y —más
importante— **qué se pierde si la ordenas mal**.
